# Supervisor Pattern

**Pattern**: A central supervisor agent coordinates specialized worker agents, controlling communication flow and task delegation.

**Architecture**:
```
                    ┌→ [Research Agent] ─┐
User → [Supervisor] ┼→ [Math Agent]     ─┼→ [Supervisor] → User
                    └→ [Writing Agent]  ─┘
```

**Key Concepts**:
- Supervisor decides which agent to invoke based on task
- Agents communicate via handoff tools
- Supervisor aggregates results and responds to user

## Setup

In [ ]:
# Install if needed
# !pip install langgraph-supervisor langchain-aws

In [ ]:
from typing import Annotated
from langchain_core.tools import tool
from langchain_aws import ChatBedrockConverse
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

## Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    temperature=0,
)

## Define Tools for Specialized Agents

In [ ]:
# Math tools
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        return "Error: Division by zero"
    return a / b

math_tools = [add, subtract, multiply, divide]

In [ ]:
# Research tools (simulated)
@tool
def search_web(query: str) -> str:
    """Search the web for information on a topic."""
    # Simulated search results
    results = {
        "python": "Python is a high-level programming language known for its simplicity and readability. Created by Guido van Rossum in 1991.",
        "langgraph": "LangGraph is a framework for building stateful, multi-step AI applications using a graph-based approach.",
        "machine learning": "Machine learning is a subset of AI that enables systems to learn from data without explicit programming.",
    }
    query_lower = query.lower()
    for key, value in results.items():
        if key in query_lower:
            return value
    return f"Search results for '{query}': This is a general topic. Please consult specialized resources for detailed information."

@tool
def get_current_date() -> str:
    """Get the current date."""
    from datetime import date
    return str(date.today())

research_tools = [search_web, get_current_date]

In [ ]:
# Writing tools
@tool
def format_as_bullet_points(text: str) -> str:
    """Format text as bullet points."""
    sentences = text.split(". ")
    bullets = "\n".join(f"- {s.strip()}" for s in sentences if s.strip())
    return bullets

@tool
def count_words(text: str) -> int:
    """Count the number of words in text."""
    return len(text.split())

@tool
def summarize_text(text: str, max_words: int = 50) -> str:
    """Summarize text to approximately max_words."""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + "..."

writing_tools = [format_as_bullet_points, count_words, summarize_text]

## Create Specialized Agents

Each agent is a ReAct agent with specific tools and expertise.

In [ ]:
# Math Expert Agent
math_agent = create_react_agent(
    model=llm,
    tools=math_tools,
    name="math_expert",
    prompt="You are a math expert. Use your tools to perform calculations accurately. Always show your work."
)

# Research Expert Agent
research_agent = create_react_agent(
    model=llm,
    tools=research_tools,
    name="research_expert",
    prompt="You are a research expert. Use your tools to find information and provide accurate, well-sourced answers."
)

# Writing Expert Agent
writing_agent = create_react_agent(
    model=llm,
    tools=writing_tools,
    name="writing_expert",
    prompt="You are a writing expert. Use your tools to format, analyze, and improve text content."
)

## Create Supervisor

The supervisor coordinates the specialized agents.

In [ ]:
supervisor_prompt = """You are a team supervisor managing three specialized agents:

1. math_expert: Handles all mathematical calculations (addition, subtraction, multiplication, division)
2. research_expert: Handles information lookup, web searches, and factual queries
3. writing_expert: Handles text formatting, summarization, and word counting

Your job is to:
- Analyze the user's request
- Delegate tasks to the appropriate expert(s)
- Combine results if multiple experts are needed
- Provide a clear, helpful final response

Always delegate to the most appropriate expert. Do not try to answer questions yourself if an expert can handle it better."""

# Create supervisor workflow
workflow = create_supervisor(
    agents=[math_agent, research_agent, writing_agent],
    model=llm,
    prompt=supervisor_prompt,
    output_mode="full_history"  # Include all messages from agent interactions
)

# Compile the workflow
app = workflow.compile()

## Visualize the Supervisor Graph

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## Helper: Display Results

In [ ]:
def display_result(result, title=""):
    """Display the result in a formatted way."""
    print("=" * 70)
    if title:
        print(f"  {title}")
        print("=" * 70)
    
    messages = result.get("messages", [])
    
    for msg in messages:
        role = getattr(msg, "type", "unknown")
        name = getattr(msg, "name", None)
        content = getattr(msg, "content", str(msg))
        
        if role == "human":
            print(f"\n[USER]")
            print(f"  {content}")
        elif role == "ai":
            agent_name = name or "Assistant"
            if content:  # Only print if there's content
                print(f"\n[{agent_name.upper()}]")
                print(f"  {content[:500]}{'...' if len(content) > 500 else ''}")
        elif role == "tool":
            print(f"\n  [Tool: {name}] {content[:200]}{'...' if len(str(content)) > 200 else ''}")
    
    print("\n" + "=" * 70)

## Test Case 1: Math Query

In [ ]:
result = app.invoke({
    "messages": [("user", "What is 25 multiplied by 4, then add 50?")]
})

display_result(result, "TEST 1: Math Query")

## Test Case 2: Research Query

In [ ]:
result = app.invoke({
    "messages": [("user", "What is LangGraph and when was Python created?")]
})

display_result(result, "TEST 2: Research Query")

## Test Case 3: Writing Query

In [ ]:
result = app.invoke({
    "messages": [("user", "Format this as bullet points: AI is transforming industries. Machine learning enables prediction. Deep learning powers image recognition.")]
})

display_result(result, "TEST 3: Writing Query")

## Test Case 4: Multi-Agent Query

A query that requires multiple agents to work together.

In [ ]:
result = app.invoke({
    "messages": [("user", "Search for information about machine learning, then count how many words are in the result.")]
})

display_result(result, "TEST 4: Multi-Agent Query")

## Test Case 5: Complex Multi-Step Query

In [ ]:
result = app.invoke({
    "messages": [("user", "Calculate 100 divided by 4, then multiply by 3. Also tell me what today's date is.")]
})

display_result(result, "TEST 5: Complex Multi-Step Query")

---
## Supervisor with Memory (Persistence)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Create checkpointer for persistence
checkpointer = InMemorySaver()

# Compile with checkpointer
persistent_app = workflow.compile(checkpointer=checkpointer)

In [ ]:
# Multi-turn conversation with memory
config = {"configurable": {"thread_id": "conversation-1"}}

# Turn 1
result1 = persistent_app.invoke(
    {"messages": [("user", "Calculate 50 times 2")]},
    config=config
)
print("Turn 1 - User: Calculate 50 times 2")
print(f"Response: {result1['messages'][-1].content[:200]}...")

# Turn 2 - References previous calculation
result2 = persistent_app.invoke(
    {"messages": [("user", "Now add 25 to that result")]},
    config=config
)
print("\nTurn 2 - User: Now add 25 to that result")
print(f"Response: {result2['messages'][-1].content[:200]}...")

---
## Stream Supervisor Execution

In [ ]:
print("=" * 70)
print("STREAMING SUPERVISOR EXECUTION")
print("=" * 70)

for chunk in app.stream(
    {"messages": [("user", "What is Python? Also calculate 10 + 20.")]}
):
    for node_name, output in chunk.items():
        print(f"\n[NODE: {node_name}]")
        if "messages" in output:
            last_msg = output["messages"][-1] if output["messages"] else None
            if last_msg:
                content = getattr(last_msg, "content", str(last_msg))
                if content:
                    print(f"  {content[:150]}{'...' if len(content) > 150 else ''}")

---
## Custom Supervisor (Manual Implementation)

For more control, you can implement the supervisor pattern manually.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# State
class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str

# Supervisor decides which agent to call
def supervisor_node(state: SupervisorState) -> dict:
    last_message = state["messages"][-1].content.lower()
    
    # Simple routing logic
    if any(word in last_message for word in ["calculate", "math", "add", "multiply", "divide", "subtract"]):
        return {"next_agent": "math"}
    elif any(word in last_message for word in ["search", "find", "what is", "who is", "when"]):
        return {"next_agent": "research"}
    elif any(word in last_message for word in ["format", "summarize", "bullet", "count words"]):
        return {"next_agent": "writing"}
    else:
        return {"next_agent": "done"}

def route_to_agent(state: SupervisorState) -> Literal["math", "research", "writing", "done"]:
    return state["next_agent"]

# Agent nodes (simplified)
def math_node(state: SupervisorState) -> dict:
    response = llm.invoke([{"role": "user", "content": f"You are a math expert. Answer: {state['messages'][-1].content}"}])
    return {"messages": [("assistant", f"[Math Expert] {response.content}")]}

def research_node(state: SupervisorState) -> dict:
    response = llm.invoke([{"role": "user", "content": f"You are a research expert. Answer: {state['messages'][-1].content}"}])
    return {"messages": [("assistant", f"[Research Expert] {response.content}")]}

def writing_node(state: SupervisorState) -> dict:
    response = llm.invoke([{"role": "user", "content": f"You are a writing expert. Answer: {state['messages'][-1].content}"}])
    return {"messages": [("assistant", f"[Writing Expert] {response.content}")]}

# Build graph
manual_builder = StateGraph(SupervisorState)

manual_builder.add_node("supervisor", supervisor_node)
manual_builder.add_node("math", math_node)
manual_builder.add_node("research", research_node)
manual_builder.add_node("writing", writing_node)

manual_builder.add_edge(START, "supervisor")
manual_builder.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "math": "math",
        "research": "research",
        "writing": "writing",
        "done": END
    }
)

# All agents return to END
manual_builder.add_edge("math", END)
manual_builder.add_edge("research", END)
manual_builder.add_edge("writing", END)

manual_supervisor = manual_builder.compile()

In [ ]:
# Test manual supervisor
result = manual_supervisor.invoke({
    "messages": [("user", "Calculate 15 times 8")]
})

print("=" * 70)
print("MANUAL SUPERVISOR TEST")
print("=" * 70)
print(f"\nRouted to: {result['next_agent']}")
print(f"Response: {result['messages'][-1].content}")

## Key Takeaways

1. **Supervisor Pattern**: Central agent coordinates specialized worker agents
2. **create_supervisor()**: High-level API from langgraph-supervisor package
3. **Handoff Tools**: Agents communicate via tool-based handoffs
4. **Output Modes**: 
   - `full_history`: Include all agent messages
   - `last_message`: Only final response
5. **Persistence**: Add checkpointer for multi-turn conversations
6. **Manual Implementation**: More control but more code

### When to Use Supervisor Pattern

| Use Case | Supervisor Pattern? |
|----------|--------------------|
| Multiple specialized agents | Yes |
| Complex task delegation | Yes |
| Need central coordination | Yes |
| Simple linear workflow | No (use basic graph) |
| Peer-to-peer agent communication | No (use swarm pattern) |